# PRIMA NER — Run 3

## Objectif de ce run

Ce run met en œuvre les recommandations de l'encadrant après l'analyse du Run 2 :

| # | Modèle | Stratégie | Données |
|---|--------|-----------|----------|
| 1 | XLM-RoBERTa + couche de classification | Encodeur **gelé** (frozen), seule la tête NER est entraînée | **938 entrées** (train 657 / val 188 / test 93) |
| 2 | BiLSTM-CRF from scratch | Entraîné depuis zéro, sans pré-entraînement | **938 entrées** (même split) |

### Pourquoi ces choix ? / 为什么这样选择？

**XLM-RoBERTa gelé (frozen encoder)** :
- Le Run 2 a montré que fine-tuner *tous* les paramètres d'XLM-RoBERTa sur seulement 657 exemples d'entraînement conduit à une convergence lente (20 epochs nécessaires) et un risque de sur-apprentissage.
- La recommandation de l'encadrant est de ne pas ré-entraîner XLM-RoBERTa directement, mais d'utiliser ses représentations comme extracteur de traits fixes, et d'ajouter une **couche de classification des entités nommées** entraînable au-dessus.
- Avantage : seule la tête linéaire (768 → 11) est entraînée (~8 448 paramètres), ce qui réduit le risque de sur-apprentissage et accélère la convergence.

> Run 2 的实验表明，对 XLM-RoBERTa 的全部参数进行微调（fine-tuning），即使有 657 条训练数据，仍然收敛较慢（需要 20 个 epoch），且存在过拟合风险。导师建议不直接重新训练 XLM-RoBERTa，而是将其作为**固定特征提取器**（frozen），在其上方添加一个可训练的**命名实体分类层**。优势：只需训练线性分类头（768 → 11，约 8,448 个参数），收敛更快，过拟合风险更低。

---

**BiLSTM-CRF from scratch** :
- Entraîné depuis zéro (initialisation aléatoire) sur l'intégralité des 657 exemples d'entraînement.
- Constitue une baseline solide sans pré-entraînement, comparable aux systèmes classiques de NER.

> 从零开始训练（随机初始化），使用全部 657 条训练数据。作为无预训练的经典 NER 基线系统，便于与 XLM-RoBERTa 进行对比。

---

**Utilisation de toutes les données (938 entrées)** :
- Run 2 : 100 entrées annotées (gold standard uniquement), split 70 / 20 / 10.
- Run 3 : 938 entrées (100 gold + 838 restantes annotées), split **657 / 188 / 93** (70/20/10 %, seed=42).
- Avec 93 entrées de test (vs 10 en Run 2), les métriques finales sont beaucoup plus fiables et représentatives.

> Run 2 仅使用 100 条人工标注数据（gold standard），其中测试集只有 10 条，评估结果不够稳定。Run 3 整合全部 938 条标注数据（100 条 gold + 838 条剩余标注），按 70/20/10 划分为训练集 657 条 / 验证集 188 条 / 测试集 93 条（seed=42）。测试集扩大到 93 条，评估指标更加可靠。

---

### Hyperparamètres choisis / 超参数选择

Basés sur l'analyse du Run 2 :

- **XLM-RoBERTa** : encodeur gelé → seule la tête linéaire apprend → `epochs=10`, `lr=5e-4`, `batch=16`, `max_len=128`.
- **BiLSTM-CRF** : `epochs=30` (657 exemples d'entraînement → converge plus vite qu'avec 70), `lr=0.001` (meilleure généralisation que `lr=0.01` en Run 2), `batch=16`.

> 基于 Run 2 的分析结果：
> - **XLM-RoBERTa**（编码器冻结）：只训练分类头，收敛快 → `epochs=10`，`lr=5e-4`（无灾难性遗忘风险，可用更大学习率），`batch=16`，`max_len=128`。
> - **BiLSTM-CRF**：训练数据从 70 条增至 657 条，收敛更快 → `epochs=30`；`lr=0.001` 在 Run 2 测试集上 F1=0.60，优于文章参数 `lr=0.01`（F1=0.52），保持不变；`batch=16`。

---

Exécuter **Cell 0** en premier : installation des dépendances + **upload des 3 fichiers JSON** (`train_all.json`, `val_all.json`, `test_all.json`).

> 请先运行 **Cell 0**：安装依赖 + 上传 3 个 JSON 文件（`train_all.json`、`val_all.json`、`test_all.json`）。


In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Installation des dépendances + upload des fichiers
# À exécuter en PREMIER dans chaque nouvelle session Colab.
#
# Fichiers à uploader (3 fichiers JSON) :
#   train_all.json, val_all.json, test_all.json
# ═══════════════════════════════════════════════════════════════

import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

!pip install -q transformers seqeval scikit-learn
print('Dépendances installées.')

# ── Upload des fichiers de données ──────────────────────────────
# Uploader train_all.json, val_all.json, test_all.json
from google.colab import files
print('\nVeuillez uploader les 3 fichiers : train_all.json, val_all.json, test_all.json')
uploaded = files.upload()

# Vérification des fichiers uploadés
import os
for fname in ['train_all.json', 'val_all.json', 'test_all.json']:
    if fname in uploaded:
        size = len(uploaded[fname])
        print(f'  ✓ {fname} ({size/1024:.0f} KB)')
    else:
        print(f'  ✗ {fname} MANQUANT — relancer cette cellule')

# Écrire les fichiers uploadés dans /content/ (répertoire de travail Colab)
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
print('\nFichiers disponibles dans /content/')


GPU disponible : True
GPU : Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Dépendances installées.

Veuillez uploader les 3 fichiers : train_all.json, val_all.json, test_all.json


Saving train_all.json to train_all.json
Saving test_all.json to test_all.json
Saving val_all.json to val_all.json
  ✓ train_all.json (1997 KB)
  ✓ val_all.json (561 KB)
  ✓ test_all.json (285 KB)

Fichiers disponibles dans /content/


## Cell 1 — XLM-RoBERTa avec couche de classification (encodeur gelé)

**Stratégie** : XLM-RoBERTa est utilisé comme **extracteur de représentations fixe** (frozen). Seule la couche linéaire de classification NER est entraînée.

**Hyperparamètres** :
- `epochs=10` : encodeur gelé → seule la tête linéaire apprend, convergence rapide.
- `lr=5e-4` : plus élevé qu'en fine-tuning complet (2e-5) car pas de risque de catastrophic forgetting (encodeur gelé).
- `batch=16` : corpus plus grand (657 train) → batch plus grand possible.
- `max_len=128` : validé par le Run 2.

**Données** : `train_all.json` (657) / `val_all.json` (188) / `test_all.json` (93).

**Sorties** → `/content/run3_xlmroberta_frozen/`

Cell 1 — XLM-RoBERTa + 分类层（编码器冻结）

策略：XLM-RoBERTa 作为固定特征提取器（frozen）使用，编码器权重不更新。仅训练上方的线性 NER 分类层。

超参数：

epochs=10：编码器已冻结，只有线性分类头在学习，收敛快，10 个 epoch 已足够。

lr=5e-4：高于完整微调时的学习率（2e-5）。因为编码器被冻结，不存在破坏预训练权重的风险（catastrophic forgetting），可以使用更大的学习率。

batch=16：训练集扩大到 657 条，可以使用更大的 batch size。

max_len=128：Run 2 已验证，我们的条目在 128 个 token 以内。

数据：train_all.json（657 条）/ val_all.json（188 条）/ test_all.json（93 条）。

输出 → /content/run3_xlmroberta_frozen/


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — XLM-RoBERTa (encodeur GELÉ + couche de classification)
# Split fixe : train(657) / val(188) / test(93)
#
# Stratégie (recommandation encadrant) :
#   - Encodeur XLM-RoBERTa gelé (requires_grad=False).
#   - Seule la couche linéaire de classification (768 → 11) est entraînée.
#   - Avec 938 entrées, un split fixe 70/20/10 suffit (93 test >> 10 en Run 2).
#
# Hyperparamètres :
#   EPOCHS=10   : tête linéaire uniquement → convergence rapide
#   LR=5e-4     : pas de risque sur l'encodeur gelé (Run 2 : 2e-5 pour fine-tuning complet)
#   BATCH=16    : corpus plus grand → batch plus grand
#   MAX_LEN=128 : validé par le Run 2
#
# Données : train_all.json / val_all.json / test_all.json (uploadés via Cell 0)
# Sorties : /content/run3_xlmroberta_frozen/
# ═══════════════════════════════════════════════════════════════

import json, os, time
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME = "xlm-roberta-base"
DATA_DIR   = "/content"  # fichiers uploadés via Cell 0
LOG_DIR    = "/content/run3_xlmroberta_frozen/logs"
MODEL_DIR  = "/content/run3_xlmroberta_frozen/model"

EPOCHS     = 10      # encodeur gelé → seule la tête linéaire apprend
BATCH_SIZE = 16      # corpus plus grand (657 train) → batch plus grand
LR         = 5e-4   # pas de catastrophic forgetting (encodeur gelé)
MAX_LEN    = 128     # validé par le Run 2
SEED       = 42

LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # = 11

os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device : {device}")
print(f"[Config] Modèle : {MODEL_NAME} (encodeur GELÉ)")
print(f"[Config] Epochs={EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, MaxLen={MAX_LEN}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """
    Identique au Run 2 : segmentation par espaces + alignement sur le
    premier caractère de chaque mot.
    """
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1: continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"
        tokens, labels = [], []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)
        samples.append((tokens, labels))
    return samples


# ── 2. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """Alignement subword → word via word_ids() (identique Run 2)."""
    def __init__(self, samples, tokenizer, max_len):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        tokens, labels = self.samples[idx]
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        word_ids  = encoding.word_ids()
        label_ids = []
        prev_word = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(LABEL2ID[labels[wid]])
            else:
                l = labels[wid]
                if l.startswith("B-"): l = "I-" + l[2:]
                label_ids.append(LABEL2ID.get(l, 0))
            prev_word = wid
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


# ── 3. Modèle : XLM-RoBERTa gelé + tête de classification ────────────────────
class FrozenXLMRoBERTaNER(nn.Module):
    """
    XLM-RoBERTa-base (encodeur gelé) + Linear(768 → 11).
    Seule la couche linéaire est entraînable (~8 448 paramètres).
    """
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        # Gel de l'encodeur
        for param in self.encoder.parameters():
            param.requires_grad = False
        hidden_size     = self.encoder.config.hidden_size  # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss, logits


# ── 4. Évaluation ─────────────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text = [tok]; cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            lab       = batch["labels"].to(device)
            _, logits = model(input_ids=input_ids, attention_mask=attn_mask)
            preds     = torch.argmax(logits, dim=-1)
            for pred_seq, true_seq in zip(preds.cpu().tolist(), lab.cpu().tolist()):
                p_tags, t_tags = [], []
                for p, t in zip(pred_seq, true_seq):
                    if t == -100: continue
                    p_tags.append(ID2LABEL[p]); t_tags.append(ID2LABEL[t])
                all_preds.append(p_tags); all_trues.append(t_tags)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })
    return {"f1": f1, "precision": pre, "recall": rec, "report": report, "details": details}


# ── 5. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}_all.json")
    with open(path, encoding="utf-8") as f:
        globals()[f"{split}_entries"] = json.load(f)
    print(f"  {split}: {len(globals()[f'{split}_entries'])} entrées")

train_samples = json_to_bio(train_entries)
val_samples   = json_to_bio(val_entries)
test_samples  = json_to_bio(test_entries)


# ── 6. Tokenizer & DataLoaders ────────────────────────────────────────────────
print("\n[2/5] Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(NERDataset(train_samples, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NERDataset(val_samples,   tokenizer, MAX_LEN), batch_size=BATCH_SIZE)
test_loader  = DataLoader(NERDataset(test_samples,  tokenizer, MAX_LEN), batch_size=BATCH_SIZE)


# ── 7. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Initialisation du modèle...")
model = FrozenXLMRoBERTaNER(MODEL_NAME, NUM_LABELS).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"  Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

optimizer   = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)


# ── 8. Entraînement ───────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            loss, _   = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val")

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, "model.pt"))
            tokenizer.save_pretrained(MODEL_DIR)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        logf.write(json.dumps({
            "epoch": epoch, "timestamp": timestamp,
            "duration_s": round(epoch_time, 2),
            "train_loss": round(avg_loss, 6),
            "val_f1":     round(val_metrics["f1"], 6),
        }, ensure_ascii=False) + "\n")
        logf.flush()

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | Meilleur val F1 = {best_f1:.4f}")


# ── 9. Évaluation finale sur val + test ──────────────────────────────────────
print("\n[5/5] Évaluation finale...")
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "model.pt"), map_location=device))
val_metrics  = evaluate(model, val_loader,  "val",  entries=val_entries)
test_metrics = evaluate(model, test_loader, "test", entries=test_entries)

results = {
    "model": "XLM-RoBERTa frozen encoder + classification head",
    "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
    "best_val_f1": round(best_f1, 6),
    "val_metrics":  {"f1": round(val_metrics["f1"], 6),
                     "precision": round(val_metrics["precision"], 6),
                     "recall":    round(val_metrics["recall"], 6)},
    "val_report":   val_metrics["report"],
    "val_details":  val_metrics["details"],
    "test_metrics": {"f1": round(test_metrics["f1"], 6),
                     "precision": round(test_metrics["precision"], 6),
                     "recall":    round(test_metrics["recall"], 6)},
    "test_report":  test_metrics["report"],
    "test_details": test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

print(f"\n[Done] Val  F1 = {val_metrics['f1']:.4f}")
print(f"       Test F1 = {test_metrics['f1']:.4f}")
print(f"       Logs    → {LOG_DIR}/test_results.json")


[Config] Device : cuda
[Config] Modèle : xlm-roberta-base (encodeur GELÉ)
[Config] Epochs=10, LR=0.0005, Batch=16, MaxLen=128

[1/5] Chargement des données...
  train: 657 entrées
  val: 188 entrées
  test: 93 entrées

[2/5] Chargement du tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...


[3/5] Initialisation du modèle...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Paramètres entraînables : 8,459 / 278,052,107 (0.00%)

[4/5] Entraînement (10 epochs)...

Epoch 1/10  loss=2.5220  time=5.2s  [2026-06-16 05:40:35]
  [val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       162
        date       0.00      0.00      0.00       186
        etat       0.00      0.00      0.00        30
    material       0.00      0.00      0.00       187
     ouvrage       0.00      0.00      0.00       225

   micro avg       0.00      0.00      0.00       790
   macro avg       0.00      0.00      0.00       790
weighted avg       0.00      0.00      0.00       790


Epoch 2/10  loss=1.9401  time=4.2s  [2026-06-16 05:40:40]
  [val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       162
        date       0.00      0.00      0.00       186
        etat       0.00      0.00      0.00        30
    mater

## Cell 2 — BiLSTM-CRF from scratch

**Stratégie** : BiLSTM-CRF entraîné **depuis zéro** (initialisation aléatoire), sans pré-entraînement.

**Hyperparamètres** :
- `epochs=30` : Run 2 a montré que epochs=50/lr=0.001 généralise mieux qu'epochs=5/lr=0.01. Avec 657 exemples d'entraînement (vs 70), convergence plus rapide → 30 epochs.
- `lr=0.001` : Run 2 test F1=0.60 (lr=0.001) > F1=0.52 (lr=0.01 article).
- `batch=16` : corpus plus grand.

**Données** : `train_all.json` (657) / `val_all.json` (188) / `test_all.json` (93).

**Sorties** → `/content/run3_bilstm_crf/`


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — BiLSTM-CRF FROM SCRATCH
# Split fixe : train(657) / val(188) / test(93)
#
# Stratégie (recommandation encadrant) :
#   Entraîner BiLSTM-CRF depuis zéro, sans pré-entraînement.
#   Baseline classique NER sans transfert.
#
# Hyperparamètres :
#   EPOCHS=30   : Run 2 (epochs=50, lr=0.001) > article (epochs=5, lr=0.01)
#                  Avec 657 train (vs 70), convergence plus rapide → 30 suffit
#   LR=0.001    : Run 2 test F1=0.60 vs lr=0.01 → F1=0.52
#   BATCH=16    : corpus plus grand
#
# Données : train_all.json / val_all.json / test_all.json (uploadés via Cell 0)
# Sorties : /content/run3_bilstm_crf/
# ═══════════════════════════════════════════════════════════════

import json, os, time
from datetime import datetime
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR   = "/content"  # fichiers uploadés via Cell 0
LOG_DIR    = "/content/run3_bilstm_crf/logs"
MODEL_DIR  = "/content/run3_bilstm_crf/model"

EPOCHS     = 30      # Run 2 : epochs=50 (nos params) > epochs=5 (article); 657 train → 30 suffit
BATCH_SIZE = 16      # corpus plus grand
LR         = 0.001   # Run 2 : lr=0.001 → test F1=0.60 vs lr=0.01 → F1=0.52
EMBED_DIM  = 100
HIDDEN_DIM = 256
DROPOUT    = 0.3
SEED       = 42

LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device : {device}")
print(f"[Config] BiLSTM-CRF from scratch")
print(f"[Config] Epochs={EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, Embed={EMBED_DIM}, Hidden={HIDDEN_DIM}")


# ── 1. JSON → BIO ─────────────────────────────────────────────────────────────
def json_to_bio(entries):
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1: continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"
        tokens, labels = [], []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)
        samples.append((tokens, labels))
    return samples


# ── 2. Vocabulaire ────────────────────────────────────────────────────────────
def build_vocab(samples):
    freq = defaultdict(int)
    for tokens, _ in samples:
        for tok in tokens: freq[tok] += 1
    word2id = {"<PAD>": 0, "<UNK>": 1}
    for word in sorted(freq): word2id[word] = len(word2id)
    return word2id


# ── 3. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    def __init__(self, samples, word2id):
        self.data = samples; self.word2id = word2id
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        return (torch.tensor([self.word2id.get(t, 1) for t in tokens], dtype=torch.long),
                torch.tensor([LABEL2ID[l] for l in labels], dtype=torch.long))

def collate_fn(batch):
    token_seqs, label_seqs = zip(*batch)
    max_len   = max(len(s) for s in token_seqs)
    token_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    label_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask      = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (toks, labs) in enumerate(zip(token_seqs, label_seqs)):
        n = len(toks)
        token_ids[i, :n] = toks; label_ids[i, :n] = labs; mask[i, :n] = True
    return token_ids, label_ids, mask


# ── 4. CRF ────────────────────────────────────────────────────────────────────
class CRF(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.num_labels        = num_labels
        self.transitions       = nn.Parameter(torch.randn(num_labels, num_labels))
        self.start_transitions = nn.Parameter(torch.randn(num_labels))
        self.end_transitions   = nn.Parameter(torch.randn(num_labels))

    def _forward_alg(self, emissions, mask):
        score = self.start_transitions + emissions[:, 0]
        for i in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + self.transitions + emissions[:, i].unsqueeze(1)
            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[:, i].unsqueeze(1), next_score, score)
        return torch.logsumexp(score + self.end_transitions, dim=1)

    def _score_sentence(self, emissions, labels, mask):
        score = self.start_transitions[labels[:, 0]] + \
                emissions[:, 0].gather(1, labels[:, 0].unsqueeze(1)).squeeze(1)
        for i in range(1, emissions.size(1)):
            m = mask[:, i]
            score = score + (self.transitions[labels[:, i], labels[:, i-1]] +
                             emissions[:, i].gather(1, labels[:, i].unsqueeze(1)).squeeze(1)) * m
        last_idx    = mask.sum(dim=1) - 1
        last_labels = labels.gather(1, last_idx.unsqueeze(1).clamp(min=0)).squeeze(1)
        return score + self.end_transitions[last_labels]

    def neg_log_likelihood(self, emissions, labels, mask):
        return (self._forward_alg(emissions, mask) - self._score_sentence(emissions, labels, mask)).mean()

    def viterbi_decode(self, emissions, mask):
        viterbi = self.start_transitions + emissions[:, 0]
        backpointers = []
        for i in range(1, emissions.size(1)):
            next_scores = viterbi.unsqueeze(2) + self.transitions
            best_scores, best_labels = next_scores.max(dim=1)
            next_viterbi = best_scores + emissions[:, i]
            viterbi = torch.where(mask[:, i].unsqueeze(1), next_viterbi, viterbi)
            backpointers.append(best_labels)
        _, best_last = (viterbi + self.end_transitions).max(dim=1)
        best_path = [best_last]
        for bp in reversed(backpointers):
            best_last = bp.gather(1, best_last.unsqueeze(1)).squeeze(1)
            best_path.append(best_last)
        best_path.reverse()
        return torch.stack(best_path, dim=1)


# ── 5. BiLSTM-CRF ─────────────────────────────────────────────────────────────
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim // 2,
                                 num_layers=1, bidirectional=True, batch_first=True)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, num_labels)
        self.crf       = CRF(num_labels)

    def get_emissions(self, token_ids, mask):
        emb     = self.dropout(self.embedding(token_ids))
        lengths = mask.sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        out, _  = self.lstm(packed)
        out, _  = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        if out.size(1) < token_ids.size(1):
            pad = torch.zeros(out.size(0), token_ids.size(1)-out.size(1), out.size(2), device=out.device)
            out = torch.cat([out, pad], dim=1)
        return self.fc(self.dropout(out))

    def neg_log_likelihood(self, token_ids, label_ids, mask):
        return self.crf.neg_log_likelihood(self.get_emissions(token_ids, mask), label_ids, mask)

    def viterbi_decode(self, token_ids, mask):
        return self.crf.viterbi_decode(self.get_emissions(token_ids, mask), mask)


# ── 6. Évaluation ─────────────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text = [tok]; cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text: spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for token_ids, label_ids, mask in loader:
            token_ids = token_ids.to(device); mask = mask.to(device)
            preds = model.viterbi_decode(token_ids, mask)
            for pred_seq, true_seq, m in zip(preds.cpu().tolist(), label_ids.tolist(), mask.cpu().tolist()):
                all_preds.append([ID2LABEL[p] for p, v in zip(pred_seq, m) if v])
                all_trues.append([ID2LABEL[t] for t, v in zip(true_seq, m) if v])
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })
    return {"f1": f1, "precision": pre, "recall": rec, "report": report, "details": details}


# ── 7. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}_all.json")
    with open(path, encoding="utf-8") as f:
        globals()[f"{split}_entries"] = json.load(f)
    print(f"  {split}: {len(globals()[f'{split}_entries'])} entrées")

train_samples = json_to_bio(train_entries)
val_samples   = json_to_bio(val_entries)
test_samples  = json_to_bio(test_entries)


# ── 8. Vocabulaire & DataLoaders ──────────────────────────────────────────────
print("\n[2/5] Construction du vocabulaire...")
word2id = build_vocab(train_samples)
print(f"  Vocab size : {len(word2id)} mots")

train_loader = DataLoader(NERDataset(train_samples, word2id), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(NERDataset(val_samples,   word2id), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader  = DataLoader(NERDataset(test_samples,  word2id), batch_size=BATCH_SIZE, collate_fn=collate_fn)


# ── 9. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Initialisation du modèle BiLSTM-CRF...")
model = BiLSTM_CRF(len(word2id), EMBED_DIM, HIDDEN_DIM, NUM_LABELS, DROPOUT).to(device)
print(f"  Paramètres : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# ── 10. Entraînement ──────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        for token_ids, label_ids, mask in train_loader:
            token_ids = token_ids.to(device)
            label_ids = label_ids.to(device)
            mask      = mask.to(device)
            loss = model.neg_log_likelihood(token_ids, label_ids, mask)
            total_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val")

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, "model.pt"))
            with open(os.path.join(MODEL_DIR, "word2id.json"), "w") as f:
                json.dump(word2id, f, ensure_ascii=False)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        logf.write(json.dumps({
            "epoch": epoch, "timestamp": timestamp,
            "duration_s": round(epoch_time, 2),
            "train_loss": round(avg_loss, 6),
            "val_f1":     round(val_metrics["f1"], 6),
        }, ensure_ascii=False) + "\n")
        logf.flush()

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | Meilleur val F1 = {best_f1:.4f}")


# ── 11. Évaluation finale ─────────────────────────────────────────────────────
print("\n[5/5] Évaluation finale...")
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "model.pt"), map_location=device))
val_metrics  = evaluate(model, val_loader,  "val",  entries=val_entries)
test_metrics = evaluate(model, test_loader, "test", entries=test_entries)

results = {
    "model": "BiLSTM-CRF from scratch",
    "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
    "vocab_size": len(word2id),
    "best_val_f1": round(best_f1, 6),
    "val_metrics":  {"f1": round(val_metrics["f1"], 6),
                     "precision": round(val_metrics["precision"], 6),
                     "recall":    round(val_metrics["recall"], 6)},
    "val_report":   val_metrics["report"],
    "val_details":  val_metrics["details"],
    "test_metrics": {"f1": round(test_metrics["f1"], 6),
                     "precision": round(test_metrics["precision"], 6),
                     "recall":    round(test_metrics["recall"], 6)},
    "test_report":  test_metrics["report"],
    "test_details": test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

print(f"\n[Done] Val  F1 = {val_metrics['f1']:.4f}")
print(f"       Test F1 = {test_metrics['f1']:.4f}")
print(f"       Logs    → {LOG_DIR}/test_results.json")


[Config] Device : cuda
[Config] BiLSTM-CRF from scratch
[Config] Epochs=30, LR=0.001, Batch=16, Embed=100, Hidden=256

[1/5] Chargement des données...
  train: 657 entrées
  val: 188 entrées
  test: 93 entrées

[2/5] Construction du vocabulaire...
  Vocab size : 3375 mots

[3/5] Initialisation du modèle BiLSTM-CRF...
  Paramètres : 575,990

[4/5] Entraînement (30 epochs)...

Epoch 1/30  loss=35.5731  time=2.1s  [2026-06-16 05:46:44]
  [val] F1=0.3574  P=0.3966  R=0.3253
              precision    recall  f1-score   support

      auteur       0.29      0.33      0.31       162
        date       0.76      0.71      0.74       186
        etat       0.00      0.00      0.00        30
    material       0.82      0.37      0.51       187
     ouvrage       0.01      0.01      0.01       225

   micro avg       0.40      0.33      0.36       790
   macro avg       0.38      0.28      0.31       790
weighted avg       0.44      0.33      0.36       790

  ✓ Meilleur modèle sauvegardé (F1=0

In [4]:
import shutil
shutil.make_archive('/content/run3_results', 'zip', '/content', base_dir='.')
# 或只打包指定文件夹，比如把两个run3文件夹一起打包：
import os
os.makedirs('/content/run3_pack', exist_ok=True)
shutil.copytree('/content/run3_bilstm_crf', '/content/run3_pack/run3_bilstm_crf')
shutil.copytree('/content/run3_xlmroberta_frozen', '/content/run3_pack/run3_xlmroberta_frozen')
shutil.make_archive('/content/run3_pack', 'zip', '/content/run3_pack')

from google.colab import files
files.download('/content/run3_pack.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>